# Quick Demo: Hybrid QNN Tutorial
## Feed-Forward + Data Re-Upload Quantum Neural Networks

This tutorial demonstrates the basic usage of hybrid quantum neural networks combining:
- **Feed-forward structure**: Measurement results feed into next layer (like multi-layer NNs)
- **Data re-upload structure**: Repeated encoding to improve learning capacity

### Key Concepts:
- 🔄 **Feed-forward**: Adds non-linearity through measurements, reduces gate errors
- 📤 **Re-upload**: Encodes data multiple times for better expressivity
- ⚡ **Optimized**: Uses JAX JIT compilation for 30-50% speedup

---

## 1. Setup & Imports

In [1]:
import jax.numpy as jnp
from reupload_ff_circuit.util import *
from reupload_ff_circuit.q_functions import *
from reupload_ff_circuit.q_circuits import *
from reupload_ff_circuit.util import *
from reupload_ff_circuit.memory_monitor import *

print("✓ Libraries loaded")

✓ Libraries loaded


## 2. Define Circuit Architecture

The circuit is configured by 5 parameters:

| Parameter | Symbol | Description |
|-----------|--------|-------------|
| **Encoding number** | `n_enc` | Rotation gates for encoding (ensure `n_enc × n_q ≥ features`) |
| **Qubit number** | `n_q` | Number of qubits |
| **Feed-forward number** | `n_f` | Layers (1 = no feed-forward) |
| **Re-upload number** | `n_r` | Repetitions of encode+variational |
| **Variational number** | `n_v` | Rotation+CNOT repetitions |

In [2]:
# Circuit architecture: (n_enc, n_q, n_f, n_r, n_v)
setting = n_enc, n_q, n_f, n_r, n_v = 5, 2, 2, 3, 2

print(f"Circuit Configuration:")
print(f"  Encoding gates: {n_enc}")
print(f"  Qubits: {n_q}")
print(f"  Feed-forward layers: {n_f}")
print(f"  Re-upload repetitions: {n_r}")
print(f"  Variational repetitions: {n_v}")
n_params = (n_enc + n_v * 3) * n_q * n_r * n_f
print(f"  Parameters {setting} → {n_params} parameters")
# Create circuit
qc = qcircuit(*setting)
print(f"\n✓ Hybrid QNN circuit created")

Circuit Configuration:
  Encoding gates: 5
  Qubits: 2
  Feed-forward layers: 2
  Re-upload repetitions: 3
  Variational repetitions: 2
  Parameters (5, 2, 2, 3, 2) → 132 parameters

✓ Hybrid QNN circuit created


## 3. Initialize Parameters & Output States

**Output states** define the predefined states for classification:
- `tetrahedron`: 4 symmetric states on Bloch sphere
- `square`: 4 states
- `binary`: 2 states

**Parameters** are randomly initialized with variance scaling for better convergence.

In [3]:
# Define output quantum states (4 classes)
shape = 'tetrahedron'
state_labels, dm_labels, Yc = predefined_states_dm(shape, n_q, display=False)

print(f"Output States:")
print(f"  Shape: {shape}")
print(f"  Number of classes: {len(state_labels)}")

# Initialize parameters with He-style scaling (optimization)
params = initialize_params(*setting, len(state_labels), seed_num=42)

print(f"\nParameter Groups:")
for key, val in params.items():
    print(f"  {key}: shape {val.shape}")

print(f"\n✓ Parameters initialized")

theta = [0.         1.91063324 1.91063324 1.91063324] 
phi = [0.         2.0943951  4.1887902  6.28318531]
c_states= [[[ 1.        +0.0000000e+00j]
  [ 0.        +0.0000000e+00j]]

 [[ 0.57735026+0.0000000e+00j]
  [-0.4082483 +7.0710677e-01j]]

 [[ 0.57735026+0.0000000e+00j]
  [-0.4082483 -7.0710677e-01j]]

 [[ 0.57735026+0.0000000e+00j]
  [ 0.8164966 -1.9998399e-16j]]] 
shape: (4, 2, 1)
Output States:
  Shape: tetrahedron
  Number of classes: 4

Parameter Groups:
  scaling: shape (2, 3, 2, 5)
  circ: shape (2, 3, 2, 2, 3)
  loss: shape (2, 4)

✓ Parameters initialized


## 4. Basic Usage: Compute Circuit Output

**Input format**: `(n_samples, n_features)` → Transpose to `(n_features, n_samples)`

**Two functions available:**
- `qc_nq()`: Standard computation
- `jqc_nq()`: JIT-compiled (30-50% faster)

In [4]:
# Example data: 2 samples, 3 features
x_data = jnp.array([[1.0, 2.0, 3.0], 
                    [4.0, 5.0, 6.0]])

print(f"Input data shape: {x_data.shape}")
print(f"  (2 samples, 3 features)")

# Compute with standard method
results = qc.qc_nq(params, x_data.T, dm_labels[0])

print(f"\nOutput:")
print(f"  Type: {type(results)}")
print(f"  Length: {len(results)} (one per qubit)")
print(f"  Qubit 0 fidelities: {results[0]}")
print(f"  Qubit 1 fidelities: {results[1]}")

Input data shape: (2, 3)
  (2 samples, 3 features)

Output:
  Type: <class 'jaxlib._jax.ArrayImpl'>
  Length: 2 (one per qubit)
  Qubit 0 fidelities: [0.56172877 0.8392896 ]
  Qubit 1 fidelities: [0.39139224 0.34080019]


## 5. Memory-Efficient Processing for Large Datasets

For large datasets (>100 samples), use `jqc_nq_chunked()` to reduce memory by 50-80%:

In [5]:
# Generate LARGE dataset (memory reduction only helps with larger data)
import psutil
import gc

# Use breast_cancer dataset (30 features, more memory intensive)
X_large, y_large = initialize_data('breast_cancer', n_training=500, preprocess='scaling')

print(f"Large dataset: {X_large.shape}")
print(f"  {len(X_large)} samples, {X_large.shape[1]} features")
print(f"\n⚠️  Note: Memory benefits require:")
print(f"     1. Large sample count (>300)")
print(f"     2. Many features (breast_cancer has 30)")
print(f"     3. Deep circuits (more layers/qubits)")

# Memory comparison: standard vs chunked
process = psutil.Process()

# Test 1: Standard jqc_nq (memory-intensive)
print(f"\n{'='*60}")
print("Test 1: Standard jqc_nq()")
print('='*60)

import time
gc.collect()
jax.clear_caches()
mem_before_std = process.memory_info().rss / 1024 / 1024  # MB
start = time.perf_counter()
for i in range(1):
    results_standard = qc.jqc_nq(params, X_large.T, dm_labels[0])
end = time.perf_counter()
timing_1 = end-start

mem_after_std = process.memory_info().rss / 1024 / 1024
mem_used_std = mem_after_std - mem_before_std

print(f"  Memory before: {mem_before_std:.1f} MB")
print(f"  Memory after:  {mem_after_std:.1f} MB")
print(f"  Memory used:   {mem_used_std:+.1f} MB")

# Test 2: Chunked jqc_nq_chunked (memory-efficient)
print(f"\n{'='*60}")
print("Test 2: Chunked jqc_nq_chunked(chunk_size=50)")
print('='*60)
# del results_standard
gc.collect()
jax.clear_caches()
mem_before_chunk = process.memory_info().rss / 1024 / 1024
start = time.perf_counter()
for i in range(1):
    results_chunked = qc.jqc_nq_chunked(params, X_large.T, dm_labels[0], chunk_size=50)
end = time.perf_counter()
timing_2 = end-start

mem_after_chunk = process.memory_info().rss / 1024 / 1024
mem_used_chunk = mem_after_chunk - mem_before_chunk

print(f"  Memory before: {mem_before_chunk:.1f} MB")
print(f"  Memory after:  {mem_after_chunk:.1f} MB")
print(f"  Memory used:   {mem_used_chunk:+.1f} MB")

# Comparison
print(f"\n{'='*60}")
print("COMPARISON")
print('='*60)
if mem_used_std > 0 and mem_used_chunk > 0:
    if mem_used_chunk < mem_used_std:
        mem_reduction = (1 - mem_used_chunk / mem_used_std) * 100
        print(f"✅ Memory Reduction: {mem_reduction:.1f}%")
    else:
        mem_increase = (mem_used_chunk / mem_used_std - 1) * 100
        print(f"⚠️  Memory Increase: {mem_increase:.1f}%")
        print(f"\n💡 Why no improvement?")
        print(f"   - Dataset too small ({len(X_large)} samples)")
        print(f"   - Chunking overhead > memory savings")
        print(f"   - Try: More samples (>500) or smaller chunk_size")
    
    print(f"\n   Standard used:  {mem_used_std:.1f} MB")
    print(f"   Chunked used:   {mem_used_chunk:.1f} MB")
    print(f"   Difference:     {mem_used_std - mem_used_chunk:+.1f} MB")
    print(f"   Time comparison:     {timing_1:.2f}s vs {timing_2:.2f}s")
else:
    print(f"⚠️  Memory measurement unreliable (values too small)")

# Verify correctness
match = jnp.allclose(results_standard[:, :50], results_chunked[:, :50], rtol=1e-5)
print(f"\n✓ Results match: {match}")

print(f"\n{'='*60}")
print("WHEN TO USE CHUNKED PROCESSING")
print('='*60)
print("""
Use jqc_nq_chunked() when:
✓ Dataset has >500 samples
✓ Circuit has many layers (n_f > 2) or qubits (n_q > 3)
✓ Getting out-of-memory errors
✓ Processing very high-dimensional data

Use standard jqc_nq() when:
✓ Dataset has <200 samples
✓ Simple circuit (n_f=1-2, n_q=1-2)
✓ Memory is not a constraint
""")

Large dataset: (500, 5)
  500 samples, 5 features

⚠️  Note: Memory benefits require:
     1. Large sample count (>300)
     2. Many features (breast_cancer has 30)
     3. Deep circuits (more layers/qubits)

Test 1: Standard jqc_nq()
  Memory before: 572.2 MB
  Memory after:  615.5 MB
  Memory used:   +43.3 MB

Test 2: Chunked jqc_nq_chunked(chunk_size=50)
  Memory before: 615.1 MB
  Memory after:  622.4 MB
  Memory used:   +7.4 MB

COMPARISON
✅ Memory Reduction: 83.0%

   Standard used:  43.3 MB
   Chunked used:   7.4 MB
   Difference:     +36.0 MB
   Time comparison:     3.32s vs 3.36s

✓ Results match: True

WHEN TO USE CHUNKED PROCESSING

Use jqc_nq_chunked() when:
✓ Dataset has >500 samples
✓ Circuit has many layers (n_f > 2) or qubits (n_q > 3)
✓ Getting out-of-memory errors
✓ Processing very high-dimensional data

Use standard jqc_nq() when:
✓ Dataset has <200 samples
✓ Simple circuit (n_f=1-2, n_q=1-2)
✓ Memory is not a constraint



## 6. Optimized Computation with JIT

Use `jqc_nq()` for faster computation (compiled with JAX):

In [6]:
# Speed Demo: jqc_nq() vs qc_nq()
import time

# Setup
X_speed, y_speed = initialize_data('breast_cancer', n_training=300, preprocess='scaling')
print(f"Speed test dataset: {X_speed.shape} ({len(X_speed)} samples)")

# Warmup JIT compilation
print("\nWarming up JIT compilation...")
_ = qc.jqc_nq(params, X_speed[:10].T, dm_labels[0])
print("✓ JIT compiled")

# Test 1: Standard qc_nq()
print("\n" + "="*60)
print("Standard qc_nq() (no JIT)")
print("="*60)
times_std = []
for i in range(5):
    t0 = time.time()
    result_std = qc.qc_nq(params, X_speed.T, dm_labels[0])
    t = time.time() - t0
    times_std.append(t)
    print(f"  Run {i+1}: {t*1000:.1f} ms")
avg_std = sum(times_std) / len(times_std)

# Test 2: JIT jqc_nq()
print("\n" + "="*60)
print("JIT jqc_nq() (compiled)")
print("="*60)
times_jit = []
for i in range(5):
    t0 = time.time()
    result_jit = qc.jqc_nq(params, X_speed.T, dm_labels[0])
    t = time.time() - t0
    times_jit.append(t)
    print(f"  Run {i+1}: {t*1000:.1f} ms")
avg_jit = sum(times_jit) / len(times_jit)

# Results
speedup = avg_std / avg_jit
print("\n" + "="*60)
print("RESULTS")
print("="*60)
print(f"Standard (qc_nq):    {avg_std*1000:>7.1f} ms")
print(f"JIT (jqc_nq):        {avg_jit*1000:>7.1f} ms")
print(f"Speedup:             {speedup:>7.2f}x")
print(f"Time saved:          {(avg_std-avg_jit)*1000:>7.1f} ms per call")

match = jnp.allclose(result_std, result_jit, rtol=1e-5)
print(f"\n✓ Results match: {match}")
print(f"\n💡 Always use jqc_nq() for {speedup:.1f}x speedup!")

Speed test dataset: (300, 5) (300 samples)

Warming up JIT compilation...
✓ JIT compiled

Standard qc_nq() (no JIT)
  Run 1: 2935.5 ms
  Run 2: 1491.8 ms
  Run 3: 1513.5 ms
  Run 4: 1524.4 ms
  Run 5: 1659.6 ms

JIT jqc_nq() (compiled)
  Run 1: 3032.8 ms
  Run 2: 2.0 ms
  Run 3: 8.5 ms
  Run 4: 10.1 ms
  Run 5: 11.1 ms

RESULTS
Standard (qc_nq):     1824.9 ms
JIT (jqc_nq):          612.9 ms
Speedup:                2.98x
Time saved:           1212.0 ms per call

✓ Results match: True

💡 Always use jqc_nq() for 3.0x speedup!


## 7. Simple Training Example

Train the circuit for binary classification:

In [7]:
# Load binary classification data
X_train, y_train = initialize_data('moon', n_training=100, preprocess='scaling')

print(f"Training data:")
print(f"  Samples: {len(X_train)}")
print(f"  Features: {X_train.shape[1]}")
print(f"  Classes: {len(jnp.unique(y_train))}")

# Training uses test() function from q_circuits
# For full training loop, see Demo_script_optimized.ipynb

print(f"\n💡 For complete training example, see:")
print(f"   - Demo_script_optimized.ipynb (memory-optimized)")
print(f"   - Old_files/Demo_script.ipynb (original)")

Training data:
  Samples: 100
  Features: 2
  Classes: 2

💡 For complete training example, see:
   - Demo_script_optimized.ipynb (memory-optimized)
   - Old_files/Demo_script.ipynb (original)


## Summary

### Key Functions:

| Function | Use Case | Memory | Speed |
|----------|----------|--------|-------|
| `qc_nq()` | Small datasets, debugging | Normal | Baseline |
| `jqc_nq()` | Production, medium datasets | High | 1.3-1.5× faster |
| `jqc_nq_chunked()` | Large datasets (>100 samples) | 50-80% less | Similar to JIT |

### Workflow:

```python
# 1. Define architecture
qc = qcircuit(n_enc, n_q, n_f, n_r, n_v)

# 2. Initialize
state_labels, dm_labels, Yc = predefined_states_dm(shape, n_q)
params = initialize_params(n_enc, n_q, n_f, n_r, n_v, n_classes)

# 3. Compute
results = qc.jqc_nq(params, X.T, dm_labels[0])

# 4. For large datasets
results = qc.jqc_nq_chunked(params, X.T, dm_labels[0], chunk_size=32)
```

### Next Steps:

- 📖 Full training: `Demo_script_optimized.ipynb`
- 📊 Memory monitoring: `reupload_ff_circuit/memory_monitor.py`
- 📚 Thesis: [doi:10.6342/NTU202404165](https://drive.google.com/file/d/1yV0NOxuzr9Q0HYPzrn0tAS_NhO4z8QIa/view)

---

**Performance Tips:**
- ✅ Use `jqc_nq()` instead of `qc_nq()` for ~3x speedup
- ✅ Use `jqc_nq_chunked()` for datasets > 300 samples
- ✅ Ensure `n_enc × n_q ≥ n_features`
- ✅ Use variance-scaled initialization (already in `initialize_params()`)

**Memory Tips:**
- 🔧 Reduce `chunk_size` if out-of-memory (try 16 or 8)
- 🔧 Use `MemoryTracker` to monitor usage
- 🔧 Clear JAX cache periodically: `jax.clear_caches()`